# 02 — TF-IDF + XGBoost baseline with OOF probabilities

This notebook builds a lexical baseline. Importantly, the TF-IDF vectorizer is fitted **inside each OOF fold**, so the validation fold does not influence its vocabulary/IDF weights.

**Outputs:**
- `artifacts/tfidf_xgb_oof.csv`
- `artifacts/tfidf_xgb_test.csv`
- `artifacts/tfidf_xgb.joblib`

In [1]:
# Install/update the packages needed for this notebook.
!pip -q install -U "scikit-learn>=1.9,<2" "xgboost>=3.4,<4" "joblib>=1.4,<2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 26.3 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import json, gc, joblib
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from xgboost import XGBClassifier

PROJECT_DIR = Path("/content/drive/MyDrive/softcom-prompt-injection")
PROC_DIR = PROJECT_DIR / "data" / "processed"
ART_DIR = PROJECT_DIR / "artifacts"
ART_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_FOLDS = 3
MAX_FEATURES = 50000

# Fixed, intentionally simple baseline. Tune later only using the development set.
XGB_PARAMS = dict(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.90,
    colsample_bytree=0.90,
    min_child_weight=1,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

dev = pd.read_csv(PROC_DIR / "dev.csv")
test = pd.read_csv(PROC_DIR / "test.csv")
print("dev:", dev.shape, "test:", test.shape)

Mounted at /content/drive
dev: (824, 3) test: (207, 3)


### Vectorizer choice

Word unigrams/bigrams capture phrases such as instruction overrides, role changes, and extraction requests while the rule layer separately handles syntax/obfuscation signals. `TfidfVectorizer` converts raw documents into sparse TF-IDF features.

In [3]:
def build_vectorizer():
    return TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        lowercase=True,
        strip_accents="unicode",
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        max_features=MAX_FEATURES,
        dtype=np.float32,
    )

def build_xgb():
    return XGBClassifier(**XGB_PARAMS)

## OOF generation

For every fold, fit TF-IDF and XGBoost on the fold's training portion and produce probabilities for the held-out portion. The resulting vector is a genuine out-of-fold prediction for every development example.

In [4]:
texts = dev["text"].astype(str).tolist()
y = dev["label"].to_numpy(dtype=int)
oof = np.zeros(len(dev), dtype=np.float32)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for fold, (tr_idx, va_idx) in enumerate(skf.split(texts, y), start=1):
    print(f"\n===== Fold {fold}/{N_FOLDS} =====")
    vectorizer = build_vectorizer()
    X_tr = vectorizer.fit_transform([texts[i] for i in tr_idx])
    X_va = vectorizer.transform([texts[i] for i in va_idx])

    model = build_xgb()
    model.fit(X_tr, y[tr_idx], verbose=False)
    probs = model.predict_proba(X_va)[:, 1]
    oof[va_idx] = probs

    print("train matrix:", X_tr.shape, "valid matrix:", X_va.shape)
    print("fold F1:", f1_score(y[va_idx], probs >= 0.5))
    print("fold ROC-AUC:", roc_auc_score(y[va_idx], probs))

    del X_tr, X_va, model, vectorizer
    gc.collect()

assert np.all(oof > -1), "OOF vector was not completely populated."
print("\nOOF ROC-AUC:", roc_auc_score(y, oof))
print("OOF PR-AUC:", average_precision_score(y, oof))


===== Fold 1/3 =====
train matrix: (549, 1142) valid matrix: (275, 1142)
fold F1: 0.8160535117056856
fold ROC-AUC: 0.8976266666666667

===== Fold 2/3 =====
train matrix: (549, 1117) valid matrix: (275, 1117)
fold F1: 0.858085808580858
fold ROC-AUC: 0.9362133333333333

===== Fold 3/3 =====
train matrix: (550, 1119) valid matrix: (274, 1119)
fold F1: 0.8532423208191127
fold ROC-AUC: 0.922738255033557

OOF ROC-AUC: 0.9183697104677061
OOF PR-AUC: 0.929952050911379


## Fit the final TF-IDF + XGBoost model on all development data

This model is used to generate the untouched test predictions and is the model shipped to Streamlit.

In [5]:
final_vectorizer = build_vectorizer()
X_dev = final_vectorizer.fit_transform(dev["text"].astype(str))
X_test = final_vectorizer.transform(test["text"].astype(str))

final_model = build_xgb()
final_model.fit(X_dev, y, verbose=False)

test_probs = final_model.predict_proba(X_test)[:, 1].astype(np.float32)

bundle = {
    "vectorizer": final_vectorizer,
    "model": final_model,
    "feature_type": "word_tfidf_1_2gram",
}
joblib.dump(bundle, ART_DIR / "tfidf_xgb.joblib")

pd.DataFrame({"row_id": dev["row_id"], "tfidf_xgb_prob": oof}).to_csv(ART_DIR / "tfidf_xgb_oof.csv", index=False)
pd.DataFrame({"row_id": test["row_id"], "tfidf_xgb_prob": test_probs}).to_csv(ART_DIR / "tfidf_xgb_test.csv", index=False)

print("Saved model:", ART_DIR / "tfidf_xgb.joblib")
print("Saved OOF:", ART_DIR / "tfidf_xgb_oof.csv")
print("Saved test predictions:", ART_DIR / "tfidf_xgb_test.csv")

Saved model: /content/drive/MyDrive/softcom-prompt-injection/artifacts/tfidf_xgb.joblib
Saved OOF: /content/drive/MyDrive/softcom-prompt-injection/artifacts/tfidf_xgb_oof.csv
Saved test predictions: /content/drive/MyDrive/softcom-prompt-injection/artifacts/tfidf_xgb_test.csv


In [6]:
# OOF/test alignment checks
assert len(oof) == len(dev)
assert len(test_probs) == len(test)
assert pd.read_csv(ART_DIR / "tfidf_xgb_oof.csv")["row_id"].tolist() == dev["row_id"].tolist()
assert pd.read_csv(ART_DIR / "tfidf_xgb_test.csv")["row_id"].tolist() == test["row_id"].tolist()
print("Artifact alignment checks passed.")

Artifact alignment checks passed.
